In [1]:
import torch
import time
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
model_id = "google/translategemma-4b-it"

processor = AutoProcessor.from_pretrained(model_id)
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,        # was torch_dtype, now just dtype
    device_map="auto",
).eval()

print("TranslateGemma 4B loaded.")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

C:\Users\aksha\anaconda3\envs\lingomatch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\models--google--translategemma-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


TranslateGemma 4B loaded.
GPU memory: 6.34 GB


In [5]:
import time

test_sentences = [
    "مرحبا، كيف حالك؟",
    "اسمي محمد علي، ولدت في حلب عام 1987",
    "أسكن في شارع الرشيد رقم 42، دمشق",
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    "رقم الهوية الشخصية: 0123456789",
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    "بناءً على المادة الثانية من قانون العقوبات",
    "السيد عبد الرحمن الحسيني",
    "إن شاء الله سأصل غدا في المساء",
]

print("TranslateGemma 4B: Arabic -> German\n")
start_total = time.time()

for i, ar_text in enumerate(test_sentences, 1):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "source_lang_code": "ar",      # Modern Standard Arabic
                    "target_lang_code": "de-DE",   # German
                    "text": ar_text,
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    start = time.time()
    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    elapsed = time.time() - start

    generation = generation[0][input_len:]
    de_text = processor.decode(generation, skip_special_tokens=True)

    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text.strip()}")
    print(f"   ({elapsed:.1f}s)")
    print()

print(f"Total: {time.time() - start_total:.1f}s")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


TranslateGemma 4B: Arabic -> German



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


1. AR: مرحبا، كيف حالك؟
   DE: Hallo, wie geht es Ihnen?
   (9.4s)



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient leidet unter Bluthochdruck und Diabetes.
   (12.7s)



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches
   (7.7s)



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Haseni
   (3.9s)

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich hoffe, ich kann morgen Abend ankommen.
   (7.5s)

Total: 111.5s


In [6]:
import sacrebleu

references_for_bleu = [
    {"ar": "اسمي أحمد وأنا من سوريا", "ref_de": "Ich heiße Ahmed und ich komme aus Syrien."},
    {"ar": "أسكن في شارع الرشيد رقم 42، دمشق", "ref_de": "Ich wohne in der Rashid-Straße Nummer 42 in Damaskus."},
    {"ar": "رقم الهوية الشخصية: 0123456789", "ref_de": "Personalausweisnummer: 0123456789"},
]

hypotheses, references = [], []
for item in references_for_bleu:
    messages = [{"role": "user", "content": [{
        "type": "text", "source_lang_code": "ar", "target_lang_code": "de-DE", "text": item["ar"],
    }]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    de = processor.decode(gen[0][input_len:], skip_special_tokens=True).strip()
    hypotheses.append(de)
    references.append(item["ref_de"])
    print(f"HYP: {de}")

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"\nBLEU (TranslateGemma 4B): {bleu.score:.2f}")
print(f"BLEU (Opus-MT): 43.81 | NLLB-1.3B: 42.36 | NLLB-600M: 38.32")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


HYP: Mein Name ist Ahmad und ich komme aus Syrien.


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


HYP: Ich wohne in der Rue Rashid, Hausnummer 42, in Damaskus.
HYP: Persönliche Identifikationsnummer: 0123456789

BLEU (TranslateGemma 4B): 37.66
BLEU (Opus-MT): 43.81 | NLLB-1.3B: 42.36 | NLLB-600M: 38.32


In [7]:
import time
from PIL import Image

# Load your local Arabic document image (same one used in the cascade)
image_path = "../data/inputs/test_arabic_01.png"   # adjust to your actual path
image = Image.open(image_path).convert("RGB")
print(f"Loaded image: {image.size}")

# Build the image-translation message.
# Note: passing the PIL image object directly instead of a url.
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "source_lang_code": "ar",
                "target_lang_code": "de-DE",
                "image": image,
            }
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

print("Running end-to-end image -> German...")
start = time.time()
with torch.inference_mode():
    generation = model.generate(
        **inputs,
        max_new_tokens=512,        # higher than text test — full document
        do_sample=False,
    )
elapsed = time.time() - start

generation = generation[0][input_len:]
de_text = processor.decode(generation, skip_special_tokens=True)

print(f"\n=== TranslateGemma 4B: image -> German (end-to-end) ===")
print(f"Time: {elapsed:.1f}s\n")
print(de_text.strip())

Loaded image: (1822, 479)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Running end-to-end image -> German...

=== TranslateGemma 4B: image -> German (end-to-end) ===
Time: 139.7s

Die Gemeinschaft der Strafverfolgungsbehörden, die auf die Strafverfolgung spezialisiert sind, und die auf die Strafverfolgung spezialisierten Behörden, arbeiten zusammen, um die Sicherheit in der Gesellschaft zu gewährleisten. Sie arbeiten eng zusammen, um Kriminalität zu bekämpfen und die Sicherheit der Bürger zu gewährleisten. Diese Zusammenarbeit umfasst die gemeinsame Bekämpfung von Kriminalität, die gemeinsame Durchführung von Ermittlungen und die gemeinsame Bekämpfung von Terrorismus. Die Zusammenarbeit zwischen den Strafverfolgungsbehörden und den Strafverfolgungsbehörden ist von entscheidender Bedeutung, um die Sicherheit in der Gesellschaft zu gewährleisten. Die Zusammenarbeit zwischen den Strafverfolgungsbehörden und den Strafverfolgungsbehörden ist von entscheidender Bedeutung, um die Sicherheit in der Gesellschaft zu gewährleisten. Die Zusammenarbeit zwischen den St